<a href="https://colab.research.google.com/github/amayshah05/boltpreload.com/blob/main/lab5_starter_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AE 4321 Lab 5: Propulsion & Launch Vehicles [100 pts]

| Section | Topic | Points | Function(s) you write |
|---|---|---|---|
| 1 | Propulsion System Trade Study | 35 | `required_propellant()` |
| 2 | Electric Propulsion System Trade Study | 35 | `propellant_mass()`, `burn_time()` |
| 3 | Staging Architectures | 30 | `stage_initial_mass()` |

### How this notebook works
* You write **four short** Python functions (listed above). Everything else (tables, plots, sweeps, labels) is generated for you from **your** functions, so if your functions are right, every result in the lab will be right.
* Cells marked **"Provided code (run, do not edit)"** are already written for you. In Colab their code is hidden; just run them. Do not change them.
* Each cell where you write code is followed by a **check cell**. It must print **all checks passed** before you continue. Its test values are **not** the lab answers.
* You answer the questions by **reading the generated tables and plots**. Write your answers in the text cells marked **Your answer** (double-click to edit). You can type equations using LaTeX between dollar signs, e.g. `$m_0 = m_p + m_f$`.
* Run cells **top to bottom** (`Shift + Enter`, or *Runtime → Run all*). If you change a function, run all cells again.

### Writing your functions
* Every input is a **single number (Python `float`)**, never a list or array, so you do **not** need loops or vector math. The provided code calls your function once per point whenever it needs many values.
* Every function returns **a single `float`**.
* Use the constants defined in the setup cells (e.g. `G0`) rather than typing numbers. You can store intermediate results in variables, e.g. `R = np.exp(dv / (isp * G0))`.

**Python you will need** (nothing else is required):

| Math | Python |
|---|---|
| $a \times b$, $a / b$, $a + b$, $a - b$ | `a * b`, `a / b`, `a + b`, `a - b` |
| $a^2$ | `a ** 2` |
| $e^{x}$ | `np.exp(x)` |
| infinity | `np.inf` |
| "if $x \le 0$, return infinity" | `if x <= 0:` (new line, indented 4 spaces) `return np.inf` |
### Submission
You must convert this notebook into a PDF before submission. Below are a number of options you can use to do this. Do **NOT** upload the raw `.ipynb` file to Canvas as we will not be able to grade your submission.

 Make sure that all cells are executed in the notebook. Ensure that every deliverable is visible in the final PDF:
* Print statements can be used to provide values from the code
* Plots must be visible in the PDF. Directly printing this page will not display plots. All the subjective answers you write must be visible in the PDF.
* All TODO code and markdown cells must be populated

1. Download the `.ipynb` file by going to `File` → `Download` → `Download .ipynb`
2. Search online for a "ipynb to pdf converter" - [this](https://www.runcell.dev/tool/jupyter-to-pdf) one worked in my testing.
3. Upload the `.ipynb` file and download the resulting pdf.


In [ ]:
# @title Provided code (run, do not edit): imports and shared constants
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

G0 = 9.81   # m/s^2, standard gravity
print("Setup complete.")

---
# Section 1: Propulsion System Trade Study [35 pts]

## Problem setup

Consider a theoretical orbital insertion maneuver from a cruise phase that requires a $\Delta V$ of **4.273 km/s**. Throughout this section, the masses are defined as follows: $m_{pl} = 1{,}975$ kg is the payload mass; $m_s = 825$ kg is the structural mass of the stage, which is *not* part of the propulsion system; $m_p$ is the propellant mass; $m_{\text{inert}}$ is the inert mass of the propulsion system (tanks, casing, engine, feed system); $m_0$ is the initial mass of the stage at ignition, i.e., everything on board (payload, structure, propulsion inert mass, and propellant); and $m_f$ is the burnout mass, i.e., the stage mass after all propellant has been expended, $m_f = m_0 - m_p$. The payload and structure are both carried to burnout.

Two insertion stage designs are under consideration:

| Design | $I_{sp}$ (s) | Propulsion system fraction $f$ |
|---|---|---|
| Solid motor | 305 | 0.93 |
| Liquid bipropellant | 326 | 0.83 |

The propulsion system fraction is defined as
$$f = \frac{m_p}{m_p + m_{\text{inert}}}.$$

At most **14,500 kg** of propellant can be carried on the insertion stage. Use $g_0 = 9.81$ m/s².

For this section, the *optimal design* for a given propulsion system is the smallest propellant load that delivers the required $\Delta V$. Any propellant beyond that is margin, not a better design.

In [ ]:
# @title Provided code (run, do not edit): constants
DV_REQ = 4273.0    # m/s, required delta-V for the insertion burn
M_PL = 1975.0      # kg, payload mass
M_S = 825.0        # kg, structural mass (NOT part of the propulsion system)

DESIGNS = {"Solid":  {"isp": 305.0, "f": 0.93},
           "Liquid": {"isp": 326.0, "f": 0.83}}
print("Constants loaded.")

---
## 1.1 Mass Model and Minimum Propellant Load [13 pts]

**(a) Derivation (optional).** Write expressions for the initial mass $m_0$ and the burnout mass $m_f$ as functions of $m_p$ and $f$. Set $m_0/m_f$ equal to the mass ratio required by the rocket equation, $R = \exp\!\left(\Delta V / I_{sp}\, g_0\right)$, and solve for $m_p$ in closed form. This is the formula you will implement in (c).

> **Note:** A typed derivation is **not required**. The only required deliverable for this step is the Python function in (c). If you would like to be eligible for partial credit in case your function is incorrect, type your full derivation in the cell below.

**Your answer (optional):**

*If you would like your derivation considered for partial credit, type it here, step by step. Otherwise, you may leave this cell blank.*

**(b) Infeasibility condition.** Consider a general required $\Delta V$ (not only the mission's 4.273 km/s), with required mass ratio $R = \exp\!\left(\Delta V / I_{sp}\, g_0\right)$. Using your expression, identify the condition on $f$ and $R$ under which **no** propellant load, no matter how large, can deliver that $\Delta V$. Explain in 1–2 sentences why this happens physically.

**Your answer:**

*Write your answer here.*

**(c) Implement your result in Python.** Complete the function `required_propellant(isp, f, dv)` in the next cell.

**Output (read carefully):**
* Return the minimum propellant mass in kg.
* Return `np.inf` (infinity) **only** when the $\Delta V$ is beyond the design's physical limit, i.e., your condition from (b) says that no finite amount of propellant, however large, can deliver it. **Never return a negative mass.**
* **Do not apply the 14,500 kg propellant cap inside this function.** If the required propellant is larger than the cap, return that number anyway. The function answers "how much propellant does the physics require?"; whether that fits within the cap is a separate question you answer in the reports below.

The constants you need are `G0`, `M_PL`, and `M_S`.

In [ ]:
def required_propellant(isp, f, dv):
    """
    Minimum propellant mass [kg] needed to deliver a velocity change dv.

    INPUTS  (each one is a single Python float, NOT a list or NumPy array)
        isp : specific impulse [s]                       e.g. 305.0
        f   : propulsion system fraction [-], 0 < f < 1  e.g. 0.93
        dv  : required delta-V [m/s], dv >= 0            e.g. 4273.0

    You may also use the constants defined above: G0, M_PL, M_S.

    OUTPUT  (a single Python float)
        the minimum propellant mass m_p in kg, OR
        np.inf  if dv is beyond this design's physical limit, i.e. no finite
                amount of propellant (however large) can deliver it. This is
                the condition you found in 1.1(b).
        Do NOT apply the 14,500 kg propellant cap here: if the answer is
        larger than the cap, still return that (large) number.
        Never return a negative number.
    """
    # ---------------- YOUR CODE BELOW ----------------
    mp = None   # replace None with your closed-form expression
    #HINT: write mp in terms of (m_pl+m_s), f, R
    # -------------------------------------------------
    return mp

In [ ]:
# @title Provided code (run, do not edit): check your function
def _check_sec1():
    tests_ok = True
    def fail(msg):
        nonlocal tests_ok
        tests_ok = False
        print("✗", msg)
    try:
        out = required_propellant(300.0, 0.90, 1000.0)
    except Exception as e:
        print("✗ Your function raised an error:", repr(e)); return
    if out is None:
        print("✗ Your function returned None. Did you replace `mp = None` with your expression?"); return
    if not isinstance(out, (float, np.floating)):
        fail(f"Expected a single float, got {type(out).__name__}.")
    elif abs(out - 1186.4) > 0.5:
        fail(f"required_propellant(300, 0.90, 1000) should be about 1186.4 kg, got {out:.1f} kg.")
    out0 = required_propellant(300.0, 0.90, 0.0)
    if not (np.isfinite(out0) and abs(out0) < 1e-6):
        fail(f"Zero delta-V should need zero propellant, got {out0}.")
    out_big = required_propellant(300.0, 0.90, 5000.0)
    if not (np.isfinite(out_big) and abs(out_big - 24846.2) < 1.0):
        fail(f"required_propellant(300, 0.90, 5000) should be about 24,846 kg (above the cap, but reachable), got {out_big}. "
             "Do not apply the 14,500 kg cap inside your function.")
    out_inf = required_propellant(300.0, 0.50, 5000.0)
    if not np.isinf(out_inf):
        fail(f"required_propellant(300, 0.50, 5000) is beyond the physical limit for any propellant load and should return np.inf, got {out_inf}.")
    # never negative, anywhere in a broad range
    for isp in (250.0, 350.0, 450.0):
        for f in (0.6, 0.8, 0.95):
            for dv in (500.0, 3000.0, 6000.0, 9000.0):
                v = required_propellant(isp, f, dv)
                if v is None or (np.isfinite(v) and v < 0):
                    fail(f"Got an invalid value {v} at isp={isp}, f={f}, dv={dv}. Masses can't be negative.")
                    break
    if tests_ok:
        print("✓ All checks passed. Continue to the next cell.")
_check_sec1()

def _require_sec1():
    """Stop with a clear message if required_propellant() is not working yet."""
    try:
        v = required_propellant(300.0, 0.90, 1000.0)
        ok = v is not None and np.isfinite(v) and abs(v - 1186.4) < 0.5
    except Exception:
        ok = False
    if not ok:
        raise RuntimeError("required_propellant() is not working yet. Fix it and re-run the check cell first.")

In [ ]:
# @title Provided code (run, do not edit): results table for 1.1
_mp_vec = np.vectorize(required_propellant, otypes=[float])   # calls your function point by point

def stage_masses(mp, isp, f):
    """Forward mass model: (m0, mf, dv) for a given propellant load."""
    mp = np.asarray(mp, dtype=float)
    m_inert = mp * (1.0 / f - 1.0)
    mf = M_PL + M_S + m_inert
    m0 = mf + mp
    return m0, mf, isp * G0 * np.log(m0 / mf)

def dv_ceiling(isp, f, hi=50000.0):
    """Largest dv your function can still reach (found numerically from your function)."""
    lo = 0.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if np.isfinite(required_propellant(isp, f, mid)): lo = mid
        else: hi = mid
    return lo

_require_sec1()
rows = []
for name, d in DESIGNS.items():
    isp, f = d["isp"], d["f"]
    mp = required_propellant(isp, f, DV_REQ)
    dvmax = dv_ceiling(isp, f)
    row = {"Design": name,
           "R required": np.exp(DV_REQ / (isp * G0)),
           "R max (from your function)": np.exp(dvmax / (isp * G0)),
           "dV ceiling [m/s]": dvmax,
           "m_p [kg]": mp}
    if np.isfinite(mp):
        m0, mf, _ = stage_masses(mp, isp, f)
        row.update({"m_inert [kg]": mp * (1 / f - 1), "m_0 [kg]": float(m0), "m_f [kg]": float(mf)})
    else:
        row.update({"m_inert [kg]": np.inf, "m_0 [kg]": np.inf, "m_f [kg]": np.inf})
    rows.append(row)

results = pd.DataFrame(rows).set_index("Design")
display(results.style.format({"R required": "{:.3f}", "R max (from your function)": "{:.2f}",
                              "dV ceiling [m/s]": "{:,.0f}", "m_p [kg]": "{:,.0f}",
                              "m_inert [kg]": "{:,.0f}", "m_0 [kg]": "{:,.0f}", "m_f [kg]": "{:,.0f}"}))

**(d) Report.** Copy the key values from the table above for both designs. Then state whether each design satisfies the **14,500 kg propellant cap**, and by how much it is under or over the cap (kg).

**Your answer:**

*Write your answer here.*

---
## 1.2 $\Delta V$ vs. Propellant Load [9 pts]

The cell below sweeps $m_p$ from 0 to 20,000 kg and plots the $\Delta V$ each design achieves. The square marker on each curve is the minimum propellant your function computed for 4.273 km/s; it should sit exactly where the curve crosses the dashed line.

The **red dotted line** is the 14,500 kg propellant cap; the shaded region to its right is over the cap. The **circle** on each curve marks the $\Delta V$ that design achieves when loaded exactly to the cap. Its value is printed in the legend.

In [ ]:
# @title Provided code (run, do not edit): ΔV vs. propellant plot
_require_sec1()
MP_CAP = 14500.0   # kg, propellant cap
mp_sweep = np.linspace(0, 20000, 1001)
fig, ax = plt.subplots(figsize=(8, 5))
for (name, d), c in zip(DESIGNS.items(), ["C0", "C1"]):
    ax.plot(mp_sweep, stage_masses(mp_sweep, d["isp"], d["f"])[2] / 1000, lw=2, color=c,
            label=f'{name} ($I_{{sp}}$={d["isp"]:.0f} s, f={d["f"]})')
    mp_min = required_propellant(d["isp"], d["f"], DV_REQ)
    if np.isfinite(mp_min) and mp_min <= mp_sweep[-1]:
        ax.plot(mp_min, DV_REQ / 1000, "s", color=c, mec="k", ms=8,
                label=f"{name} minimum m_p = {mp_min:,.0f} kg")
ax.axhline(DV_REQ / 1000, color="k", ls="--", lw=1, label="Required ΔV (4.273 km/s)")

# propellant cap (shaded side is over the cap) and the ΔV each design achieves at the cap
ax.axvline(MP_CAP, color="r", ls=":", lw=1.5, label="Propellant cap (14,500 kg)")
ax.axvspan(MP_CAP, mp_sweep[-1], color="r", alpha=0.07)
for (name, d), c in zip(DESIGNS.items(), ["C0", "C1"]):
    dv_cap = float(stage_masses(MP_CAP, d["isp"], d["f"])[2])
    ax.plot(MP_CAP, dv_cap / 1000, "o", color=c, mec="r", mew=1.5, ms=8,
            label=f"{name} ΔV at cap = {dv_cap:,.0f} m/s")

ax.set_xlabel("Propellant mass $m_p$ [kg]"); ax.set_ylabel("ΔV [km/s]")
ax.set_title("Achievable ΔV vs. propellant load"); ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=9)
ax.set_xticks(np.arange(0, 20001, 2000))
plt.tight_layout(); plt.show()

**Report.**
1. From the circle markers on the plot (values in the legend), report the $\Delta V$ each design achieves when loaded to the **propellant cap**.
2. For each design, compute its margin or shortfall relative to 4.273 km/s in both m/s and percent.
3. In 1–2 sentences, state what the position of each design's square marker relative to the cap line tells you about that design.

**Your answer:**

*Write your answer here.*

---
## 1.3 Design Trade Space [13 pts]

The cell below evaluates your function on a grid of $I_{sp} \in [250, 450]$ s and $f \in [0.70, 0.97]$ and plots the minimum propellant needed for 4.273 km/s. Labeled lines show constant propellant mass (in tonnes, 1 t = 1,000 kg). Regions needing 40 t or more, including any where your function returns `np.inf`, are shown in the top color (yellow).

The **red line** is the 14,500 kg propellant cap; the hatched side of it needs more propellant than the cap allows. The two **red stars** are the break-even points for the liquid design, i.e., where it first meets the cap if you move straight up from it (higher $f$ at the same $I_{sp}$) or straight to the right (higher $I_{sp}$ at the same $f$). Their values are printed in the legend.

In [ ]:
# @title Provided code (run, do not edit): trade-space contour plot
_require_sec1()
MP_CAP = 14500.0   # kg, propellant cap
isp_grid = np.linspace(250, 450, 201)
f_grid = np.linspace(0.70, 0.97, 136)
I, F = np.meshgrid(isp_grid, f_grid)
MP = _mp_vec(I, F, DV_REQ)

fig, ax = plt.subplots(figsize=(8.5, 6))
MPplot = np.where(np.isfinite(MP), np.minimum(MP, 40000), 40000) / 1000
cf = ax.contourf(I, F, MPplot, levels=np.arange(0, 41, 2), cmap="viridis")
fig.colorbar(cf, ax=ax, label="Minimum propellant $m_p$ [t] (colors capped at 40 t)")
cs = ax.contour(I, F, MPplot, levels=np.arange(4, 40, 4), colors="w", linewidths=0.8)
ax.clabel(cs, fmt="%d t", fontsize=8)
for name, d in DESIGNS.items():
    ax.plot(d["isp"], d["f"], "wo", mec="k", ms=9)
    ax.annotate(name, (d["isp"], d["f"]), textcoords="offset points", xytext=(8, 6),
                color="w", fontweight="bold")

# propellant cap boundary (hatched side needs more than the cap)
MPcap = np.where(np.isfinite(MP), MP, 1e12)
ax.contourf(I, F, MPcap, levels=[MP_CAP, 2e12], colors="none", hatches=["\\\\"])
ax.contour(I, F, MPcap, levels=[MP_CAP], colors="r", linewidths=2.5)
ax.plot([], [], "r-", lw=2.5, label="14,500 kg cap boundary (hatched = over the cap)")

# break-even points for the liquid design (found numerically from your function)
def _break_even(over_cap, lo, hi):
    """Smallest value between lo and hi at which the design just meets the cap."""
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if over_cap(mid): lo = mid
        else: hi = mid
    return hi

liq = DESIGNS["Liquid"]
f_be = _break_even(lambda f: required_propellant(liq["isp"], f, DV_REQ) > MP_CAP, liq["f"], 0.9999)
isp_be = _break_even(lambda isp: required_propellant(isp, liq["f"], DV_REQ) > MP_CAP, liq["isp"], 1000.0)
ax.plot(liq["isp"], f_be, "r*", ms=13, mec="k",
        label=f'Liquid break-even: f = {f_be:.3f} at $I_{{sp}}$ = {liq["isp"]:.0f} s')
ax.plot(isp_be, liq["f"], "r*", ms=13, mec="k",
        label=f'Liquid break-even: $I_{{sp}}$ = {isp_be:.1f} s at f = {liq["f"]}')

ax.set_xlabel("$I_{sp}$ [s]"); ax.set_ylabel("Propulsion system fraction $f$")
ax.set_title("Minimum propellant to deliver ΔV = 4.273 km/s")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()

**Propellant calculator (verify the plot).** Use this cell to verify the red-star break-even values: set `ISP_TEST` and `F_TEST` to a star's values, run the cell, and check that the minimum propellant comes out at the 14,500 kg cap. Because the legend values are rounded, expect to land within about 50 kg of the cap, not exactly on it.

In [ ]:
# @title Propellant calculator (edit ISP_TEST and F_TEST only)
ISP_TEST = 326.0   # s   <-- change these and re-run
F_TEST = 0.83      # -

_require_sec1()
mp = required_propellant(ISP_TEST, F_TEST, DV_REQ)
print(f"Isp = {ISP_TEST} s, f = {F_TEST}  ->  minimum m_p = {mp:,.1f} kg")

**Report.** Items 1 and 2 are read directly from the red stars on the contour plot (their values are in the legend); no extra calculation is needed.
1. From the red star directly **above** the liquid design, report the **minimum propulsion system fraction $f$** the liquid design would need at $I_{sp} = 326$ s to meet the requirement **within the propellant cap**. Report $f$ to 3 decimal places.
2. From the red star directly **to the right** of the liquid design, report the **minimum $I_{sp}$** the liquid design would need at $f = 0.83$ to meet the requirement within the cap. Report $I_{sp}$ to 0.1 s.
3. Describe where the cap boundary (red line) lies relative to the two designs.
4. In 2–3 sentences, comment on which of the two improvements (1 or 2) seems more realistic to achieve for a liquid stage, and why.

**Your answer:**

*Write your answer here.*

---
# Section 2: Electric Propulsion System Trade Study [35 pts]

## Problem setup

A 12U CubeSat with an initial mass of **14 kg** is designed for a 2-year LEO mission at 500 km altitude. The total required $\Delta V$ is **180 m/s**. The spacecraft's propulsion power budget is capped at **30 W**, and the total propulsion system mass, $m_p + m_{\text{inert}}$, must not exceed **1.5 kg**.

Throughout this section, the masses use the same symbols as Section 1: $m_0 = 14$ kg is the initial mass of the spacecraft at the start of the mission, i.e., everything on board (bus, payload, propulsion system inert mass, and propellant), also called the *wet mass*; $m_p$ is the propellant mass; $m_{\text{inert}}$ is the inert mass of the propulsion system (the thruster hardware without propellant), listed for each system in the table below; and $m_f$ is the burnout mass, i.e., the spacecraft mass after all propellant has been expended, $m_f = m_0 - m_p$. Note that $m_{\text{inert}}$ is the mass of the propulsion hardware alone, not of the whole spacecraft, so it is **not** $m_f$.

Three candidate propulsion systems are under consideration:

| System | Classification | $I_{sp}$ (s) | Power (W) | $m_{\text{inert}}$ (kg) | Thrust (mN) |
|---|---|---|---|---|---|
| Cold gas (R-236fa) | Passive chemical | 45 | 0 | 0.20 | 15.00 |
| Pulsed Plasma (PPT) | Electromagnetic | 860 | 15 | 0.45 | 0.08 |
| Electrospray (FEEP) | Electrostatic | 2000 | 25 | 0.30 | 0.15 |

For all calculations, treat the initial spacecraft mass as $m_0 = 14$ kg. Use $g_0 = 9.81$ m/s².

In [ ]:
# @title Provided code (run, do not edit): constants and thruster data
M0 = 14.0            # kg, initial spacecraft mass m_0
DV_MISSION = 180.0   # m/s, total mission delta-V
DV_MANEUVER = 30.0   # m/s, single orbit-raising maneuver (Part 2.2)
CONTACT_MIN = 10.0   # min, ground contact window
ORBIT_MIN = 90.0     # min, orbit period at 500 km

SYSTEMS = {
    "Cold gas (R-236fa)":  {"isp": 45.0,   "power_W": 0.0,  "inert_kg": 0.20, "thrust_mN": 15.00},
    "Pulsed Plasma (PPT)": {"isp": 860.0,  "power_W": 15.0, "inert_kg": 0.45, "thrust_mN": 0.08},
    "Electrospray (FEEP)": {"isp": 2000.0, "power_W": 25.0, "inert_kg": 0.30, "thrust_mN": 0.15},
}
COLORS = {"Cold gas (R-236fa)": "C0", "Pulsed Plasma (PPT)": "C1", "Electrospray (FEEP)": "C2"}
print("Constants loaded.")

---
## Your code: two functions

**(a) Propellant mass.** Starting from the Tsiolkovsky rocket equation, $\Delta V = I_{sp}\, g_0 \ln\!\left(m_0 / m_f\right)$ with $m_f = m_0 - m_p$, solve for the propellant mass $m_p$ in terms of $I_{sp}$, $\Delta V$, and $m_0$. Write your derivation in the text cell below, then implement it as `propellant_mass(isp, dv, m0)`.

**(b) Burn time.** For a thruster producing a constant thrust $F$ on a spacecraft whose mass is treated as constant at $m_0$, the time to change velocity by $\Delta V$ is
$$t \approx \frac{m_0\, \Delta V}{F}.$$
This is the **constant-mass approximation** (it ignores the small mass lost as propellant is used). Implement it as `burn_time(m0, dv, thrust)`.

**Your answer:**

*Write your derivation of $m_p$ here, step by step.*

**Units (read carefully):**
* `propellant_mass` returns kg, `burn_time` returns **seconds**.
* `burn_time` takes thrust in **newtons**. The table above lists thrust in **millinewtons**; the provided code converts mN to N before calling your function, so do **not** convert inside your function.

In [ ]:
def propellant_mass(isp, dv, m0):
    """
    Propellant mass [kg] needed to deliver a velocity change dv.

    INPUTS  (each one is a single Python float, NOT a list or NumPy array)
        isp : specific impulse [s]                    e.g. 860.0
        dv  : required delta-V [m/s], dv >= 0         e.g. 180.0
        m0  : initial spacecraft mass m_0 [kg]        e.g. 14.0

    You may also use the constant G0 defined above.

    OUTPUT  (a single Python float)
        the propellant mass m_p in kg.
    """
    # ---------------- YOUR CODE BELOW ----------------
    mp = None   # replace None with your expression
    #HINT: Write mp in terms of m0, dv, isp, and G0.  You may use np.exp() for the exponential function.
    # -------------------------------------------------
    return mp


def burn_time(m0, dv, thrust):
    """
    Continuous burn time [s] for a maneuver, using the constant-mass approximation.

    INPUTS  (each one is a single Python float, NOT a list or NumPy array)
        m0     : initial spacecraft mass m_0 [kg], treated as constant    e.g. 14.0
        dv     : maneuver delta-V [m/s]                       e.g. 30.0
        thrust : thrust in NEWTONS [N]  (1 mN = 0.001 N)      e.g. 0.015

    OUTPUT  (a single Python float)
        the burn time in SECONDS.
    """
    # ---------------- YOUR CODE BELOW ----------------
    t = None    # replace None with your expression
    # -------------------------------------------------
    return t

In [ ]:
# @title Provided code (run, do not edit): check your functions
def _check_sec2():
    ok = True
    def fail(msg):
        nonlocal ok
        ok = False
        print("✗", msg)
    # ---- propellant_mass ----
    try:
        v = propellant_mass(300.0, 500.0, 10.0)
    except Exception as e:
        print("✗ propellant_mass raised an error:", repr(e)); return
    if v is None:
        fail("propellant_mass returned None. Did you replace `mp = None` with your expression?")
    elif not isinstance(v, (float, np.floating)):
        fail(f"propellant_mass should return a single float, got {type(v).__name__}.")
    elif abs(v - 1.8521) < 0.01:
        fail("propellant_mass(300, 500, 10) gave 1.852 kg. That treats 10 kg as the BURNOUT mass m_f. "
             "In this problem m0 is the INITIAL mass.")
    elif abs(v - 1.5625) > 0.001:
        fail(f"propellant_mass(300, 500, 10) should be about 1.5625 kg, got {v:.4f} kg.")
    else:
        v0 = propellant_mass(300.0, 0.0, 10.0)
        if abs(v0) > 1e-9:
            fail(f"Zero delta-V should need zero propellant, got {v0}.")
        vbig = propellant_mass(300.0, 20000.0, 10.0)
        if not (0 < vbig < 10.0):
            fail(f"Propellant mass must stay below m0, got {vbig} kg for m0 = 10 kg.")
    # ---- burn_time ----
    try:
        t = burn_time(10.0, 5.0, 0.01)
    except Exception as e:
        print("✗ burn_time raised an error:", repr(e)); return
    if t is None:
        fail("burn_time returned None. Did you replace `t = None` with your expression?")
    elif not isinstance(t, (float, np.floating)):
        fail(f"burn_time should return a single float, got {type(t).__name__}.")
    elif abs(t - 5.0) < 1e-6:
        fail("burn_time(10, 5, 0.01) gave 5 s. Thrust is already in newtons; do not convert it inside your function.")
    elif abs(t - 5000.0) > 0.01:
        fail(f"burn_time(10, 5, 0.01) should be 5000 s, got {t}.")
    if ok:
        print("✓ All checks passed. Continue to the next cell.")
_check_sec2()

def _require_sec2():
    try:
        good = (abs(propellant_mass(300.0, 500.0, 10.0) - 1.5625) < 0.001
                and abs(burn_time(10.0, 5.0, 0.01) - 5000.0) < 0.01)
    except Exception:
        good = False
    if not good:
        raise RuntimeError("Your functions are not working yet. Fix them and re-run the check cell first.")

---
## 2.1 Mass Calculation [10 pts]

The cell below uses your `propellant_mass()` to compute the propellant needed by each system for $\Delta V = 180$ m/s, and plots the total propulsion system mass, $m_p + m_{\text{inert}}$, as stacked bars. The number above each bar is the total.

In [ ]:
# @title Provided code (run, do not edit): propulsion system mass plot
_require_sec2()
names = list(SYSTEMS)
mp = [propellant_mass(SYSTEMS[n]["isp"], DV_MISSION, M0) for n in names]
m_inert = [SYSTEMS[n]["inert_kg"] for n in names]
m_prop_sys = [a + b for a, b in zip(mp, m_inert)]   # total propulsion system mass, m_p + m_inert

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(names))
ax.bar(x, m_inert, color="0.6", label="Propulsion system inert mass $m_{inert}$")
ax.bar(x, mp, bottom=m_inert, color=[COLORS[n] for n in names], label="Propellant mass $m_p$")
for i in range(len(names)):
    ax.text(x[i], m_prop_sys[i] + 0.08, f"total {m_prop_sys[i]:.3f} kg\n(propellant {mp[i]:.3f} kg)",
            ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("Mass [kg]"); ax.set_ylim(0, max(m_prop_sys) * 1.25)
ax.set_title("Propulsion system mass for ΔV = 180 m/s (m$_0$ = 14 kg)")
ax.grid(axis="y", alpha=0.3); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

**Report.** From the plot, report the propellant mass $m_p$ and the total propulsion system mass $m_p + m_{\text{inert}}$ for each system. In 1–2 sentences, explain why the cold gas system needs so much more propellant than the other two.

**Your answer:**

*Write your answer here.*

---
## 2.2 Burn Time & Maneuvers [11 pts]

The cell below uses your `burn_time()` to compute the minimum continuous burn time for a single **30 m/s** orbit-raising maneuver with each system. Because the times differ by several orders of magnitude, the time axis is **logarithmic** (each grid line is 10× the previous one). The dashed lines mark a 10-minute ground contact window and a 90-minute orbit period. Each bar is labeled with its time in convenient units and the equivalent number of orbits.

In [ ]:
# @title Provided code (run, do not edit): burn time plot
_require_sec2()
def _fmt(sec):
    if sec < 3600: return f"{sec/60:.1f} min"
    if sec < 86400: return f"{sec/3600:.2f} h"
    return f"{sec/86400:.1f} days"

t_s = [burn_time(M0, DV_MANEUVER, SYSTEMS[n]["thrust_mN"] / 1000.0) for n in names]
t_min = np.array(t_s) / 60.0

fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(names))
ax.barh(y, t_min, color=[COLORS[n] for n in names])
for i in range(len(names)):
    n_orb = t_min[i] / ORBIT_MIN
    orb = f"{n_orb:.1f}" if n_orb < 10 else f"{n_orb:,.0f}"
    ax.text(t_min[i] * 1.3, y[i], f"{_fmt(t_s[i])}  (≈ {orb} orbits)",
            va="center", fontsize=9)
ax.axvline(CONTACT_MIN, color="k", ls="--", lw=1.2, label="Ground contact window (10 min)")
ax.axvline(ORBIT_MIN, color="r", ls="--", lw=1.2, label="One orbit at 500 km (90 min)")
ax.set_xscale("log"); ax.set_xlim(1, max(t_min) * 40)
ax.set_yticks(y); ax.set_yticklabels(names); ax.invert_yaxis()
ax.set_xlabel("Continuous burn time [minutes] (log scale)")
ax.set_title("Burn time for a 30 m/s maneuver (constant-mass approximation)")
ax.grid(axis="x", which="major", alpha=0.4)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

**Report.** From the plot, report the burn time for each system. For each thruster, comment in 1–2 sentences on whether the maneuver is operationally practical within a 10-minute contact window or a single 90-minute orbit. If it is not, use the plot to state roughly how many orbits (or days) it would take, and what that implies for how the maneuver would have to be operated.

**Your answer:**

*Write your answer here.*

---
## 2.3 Constraint Elimination Discussion [14 pts]

The cell below places each system on a map of **power draw** vs. **total propulsion system mass** $m_p + m_{\text{inert}}$ (from your `propellant_mass()`), with the two hard constraints drawn: $m_p + m_{\text{inert}} \le 1.5$ kg and power draw $\le 30$ W. The shaded box is the region that satisfies both. The mass axis is logarithmic so that all three systems fit on one plot.

In [ ]:
# @title Provided code (run, do not edit): constraint map
_require_sec2()
MASS_MAX, POWER_MAX = 1.5, 30.0
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.fill_between([0.1, MASS_MAX], -2, POWER_MAX, color="g", alpha=0.08)
ax.axvline(MASS_MAX, color="k", ls="--", lw=1.2, label="Mass limit (1.5 kg)")
ax.axhline(POWER_MAX, color="r", ls="--", lw=1.2, label="Power limit (30 W)")
for n, m in zip(names, m_prop_sys):
    p = SYSTEMS[n]["power_W"]
    ax.plot(m, p, "o", color=COLORS[n], ms=12, mec="k")
    ax.annotate(f"{n}\n{m:.3f} kg, {p:.0f} W", (m, p), textcoords="offset points",
                xytext=(10, 8), fontsize=9)
ax.text(0.12, 1.0, "Satisfies both constraints", color="g", fontsize=10)
ax.set_xscale("log"); ax.set_xlim(0.1, 10); ax.set_ylim(-2, 40)
ax.set_xlabel("Total propulsion system mass [kg] (log scale)")
ax.set_ylabel("Power draw [W]")
ax.set_title("Hard-constraint check for ΔV = 180 m/s")
ax.grid(alpha=0.3, which="both"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

**Report.**
1. From the plot, state which system(s) survive both constraints and which constraint eliminates each of the others.
2. For the surviving candidate(s), write a **~3 sentence answer** recommending a final system selection and discussing the key tradeoffs you considered. Your answer must reference the numerical results from Parts 2.1 and 2.2.

**Your answer:**

*Write your answer here.*

---
# Section 3: Staging Architectures [30 pts]

## Problem setup

As part of the Mars Sample Return (MSR) campaign, a Mars Ascent Vehicle (MAV) must launch a **500 kg** sample-and-rendezvous capsule from the Martian surface to Low Mars Orbit (LMO) at approximately 400 km altitude, where it will dock with a waiting Earth Return Vehicle. The total $\Delta V$ required to reach LMO from the Martian surface, including gravity and atmospheric drag losses, is **3,800 m/s**. Because every kilogram of MAV hardware must itself be landed on Mars first, mass is at an absolute premium.

Both architectures below use a hypergolic liquid bipropellant engine (MMH/NTO) with $I_{sp} = 360$ s. Use $g_0 = 9.81$ m/s². For each stage, $m_s$ is its structural mass, $m_p$ its propellant mass, and the **structural mass fraction** $\varepsilon$ is defined by
$$m_s = \varepsilon\,(m_s + m_p).$$
A stage's initial mass is $m_0 = m_{\text{payload}} + m_s + m_p$, where "payload" is everything the stage carries on top of it, and its burnout mass is $m_f = m_0 - m_p$.

| Architecture | $\varepsilon$ | $\Delta V$ per stage |
|---|---|---|
| Single-stage to orbit (SSTO) | 0.15 | 3,800 m/s in one stage |
| Two-stage (ascent stage + kick stage) | 0.12 per stage | 1,900 m/s each (equal split) |

In [ ]:
# @title Provided code (run, do not edit): constants
ISP = 360.0         # s, MMH/NTO
M_CAPSULE = 500.0   # kg, sample-and-rendezvous capsule
DV_MARS = 3800.0    # m/s, Mars surface to LMO (with losses)
DV_EARTH = 9400.0   # m/s, approximate Earth surface to LEO (with losses), for comparison only
EPS_SSTO = 0.15
EPS_2STG = 0.12
print("Constants loaded.")

---
## 3.1 Feasibility Analysis [10 pts]

**(a) Derivation (optional).** For a single stage carrying a payload $m_{\text{payload}}$:
1. Use the definition of $\varepsilon$ to write $m_s$ in terms of $m_p$.
2. Write $m_0$ and $m_f$ in terms of $m_{\text{payload}}$, $m_p$, and $\varepsilon$.
3. Set $m_0 / m_f$ equal to the mass ratio required by the rocket equation, $R = \exp\!\left(\Delta V / I_{sp}\, g_0\right)$, and solve for the stage's initial mass $m_0$ in closed form.
4. Take the limit $m_{\text{payload}} \to 0$ to find the **maximum achievable** mass ratio for a given $\varepsilon$. State the condition on $R$ and $\varepsilon$ under which no stage, however large, can deliver the $\Delta V$.

*Hint:* the maximum mass ratio is not derived in lecture. Work through it from the definition of $\varepsilon$ first; if you get stuck, searching "SSTO structural fraction limit" should point you in the right direction.

> **Note:** A typed derivation is **not required**. The only required deliverable for this step is the Python function in (b), which implements the results of steps 3 and 4. If you would like to be eligible for partial credit in case your function is incorrect, type your full derivation in the cell below.

**Your answer (optional):**

*If you would like your derivation considered for partial credit, type it here, step by step. Otherwise, you may leave this cell blank.*

**(b) Implement your result in Python.** Complete `stage_initial_mass(m_payload, dv, isp, eps)` in the next cell.

**Output (read carefully):**
* Return the stage's initial mass in kg (payload + structure + propellant). The provided code calls your function **twice** (once per stage) for the two-stage vehicle.
* Return `np.inf` **only** when the $\Delta V$ is beyond the stage's physical limit, i.e., your condition from (a) says no finite amount of propellant can deliver it. **Never return a negative mass.**

In [ ]:
def stage_initial_mass(m_payload, dv, isp, eps):
    """
    Initial (liftoff) mass [kg] of ONE rocket stage that carries a payload.

    INPUTS  (each one is a single Python float, NOT a list or NumPy array)
        m_payload : mass carried on top of this stage [kg]   e.g. 500.0
        dv        : delta-V this stage must deliver [m/s]    e.g. 3800.0
        isp       : specific impulse [s]                     e.g. 360.0
        eps       : structural mass fraction [-], 0 < eps < 1  e.g. 0.15

    You may also use the constant G0 defined above.

    OUTPUT  (a single Python float)
        the stage's initial mass m0 = payload + structure + propellant, in kg, OR
        np.inf  if dv is beyond this stage's physical limit, i.e. no finite
                amount of propellant (however large) can deliver it. This is
                the condition you found in 3.1(a).
        Never return a negative number.
    """
    # ---------------- YOUR CODE BELOW ----------------
    m0 = None   # replace None with your closed-form expression
    # -------------------------------------------------
    return m0

In [ ]:
# @title Provided code (run, do not edit): check your function
def _check_sec3():
    ok = True
    def fail(msg):
        nonlocal ok
        ok = False
        print("✗", msg)
    try:
        v = stage_initial_mass(100.0, 2000.0, 300.0, 0.10)
    except Exception as e:
        print("✗ Your function raised an error:", repr(e)); return
    if v is None:
        print("✗ Your function returned None. Did you replace `m0 = None` with your expression?"); return
    if not isinstance(v, (float, np.floating)):
        fail(f"Expected a single float, got {type(v).__name__}.")
    elif abs(v - 121.22) < 0.05:
        fail("stage_initial_mass(100, 2000, 300, 0.10) gave 121.2 kg. That is the stage mass WITHOUT the payload. "
             "m0 must include the payload.")
    elif abs(v - 221.22) > 0.05:
        fail(f"stage_initial_mass(100, 2000, 300, 0.10) should be about 221.22 kg, got {v:.2f} kg.")
    v0 = stage_initial_mass(100.0, 0.0, 300.0, 0.10)
    if not (np.isfinite(v0) and abs(v0 - 100.0) < 1e-6):
        fail(f"Zero delta-V should give m0 equal to the payload (100 kg), got {v0}.")
    vinf = stage_initial_mass(100.0, 8000.0, 300.0, 0.20)
    if not np.isinf(vinf):
        fail(f"stage_initial_mass(100, 8000, 300, 0.20) is beyond the physical limit and should return np.inf, got {vinf}.")
    for eps in (0.05, 0.15, 0.3):
        for dv in (1000.0, 4000.0, 8000.0, 12000.0):
            x = stage_initial_mass(100.0, dv, 360.0, eps)
            if x is None or (np.isfinite(x) and x < 100.0):
                fail(f"Got an invalid value {x} at dv={dv}, eps={eps}. m0 can never be below the payload or negative.")
                break
    if ok:
        print("✓ All checks passed. Continue to the next cell.")
_check_sec3()

def _require_sec3():
    try:
        v = stage_initial_mass(100.0, 2000.0, 300.0, 0.10)
        good = v is not None and abs(v - 221.22) < 0.05
    except Exception:
        good = False
    if not good:
        raise RuntimeError("stage_initial_mass() is not working yet. Fix it and re-run the check cell first.")

The cell below uses your function to plot the **liftoff mass** needed to deliver the 500 kg capsule as a function of the required $\Delta V$, for both architectures (the two-stage vehicle always splits $\Delta V$ equally). The mass axis is logarithmic. The dash-dot line is the SSTO's $\Delta V$ ceiling, found numerically as the largest $\Delta V$ for which your function still returns a finite value; the label also gives the matching maximum mass ratio. The dashed lines mark the Mars ascent $\Delta V$ (with its required mass ratio) and, for comparison, an approximate Earth-to-orbit $\Delta V$ of about 9.4 km/s using the same engine and structural fractions.

In [ ]:
# @title Provided code (run, do not edit): liftoff mass vs. ΔV plot
_require_sec3()

def two_stage_mass(m_pl, dv_total, isp, eps):
    m0_2 = stage_initial_mass(m_pl, dv_total / 2, isp, eps)
    return stage_initial_mass(m0_2, dv_total / 2, isp, eps) if np.isfinite(m0_2) else np.inf

def ssto_ceiling(isp, eps, hi=50000.0):
    lo = 0.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if np.isfinite(stage_initial_mass(1.0, mid, isp, eps)): lo = mid
        else: hi = mid
    return lo

ceil = ssto_ceiling(ISP, EPS_SSTO)
R_max = np.exp(ceil / (ISP * G0))
R_mars = np.exp(DV_MARS / (ISP * G0))
R_earth = np.exp(DV_EARTH / (ISP * G0))

dv = np.unique(np.concatenate([np.linspace(1, 10000, 2500), ceil - np.logspace(-3, 3, 300)]))
dv = dv[dv > 0]
m_ssto = np.array([stage_initial_mass(M_CAPSULE, x, ISP, EPS_SSTO) for x in dv])
m_2stg = np.array([two_stage_mass(M_CAPSULE, x, ISP, EPS_2STG) for x in dv])
m_ssto = np.where(np.isfinite(m_ssto), m_ssto, np.nan)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(dv / 1000, m_ssto / 1000, lw=2, color="C0", label="SSTO (ε = 0.15)")
ax.plot(dv / 1000, m_2stg / 1000, lw=2, color="C1", label="Two-stage (ε = 0.12 each, equal ΔV split)")
ax.axvline(ceil / 1000, color="C0", ls="-.", lw=1.5,
           label=f"SSTO ΔV ceiling = {ceil/1000:.2f} km/s  (max mass ratio = {R_max:.2f})")
ax.axvline(DV_MARS / 1000, color="r", ls="--", lw=1.2,
           label=f"Mars ascent ΔV = {DV_MARS/1000:.1f} km/s  (required mass ratio = {R_mars:.2f})")
ax.axvline(DV_EARTH / 1000, color="k", ls="--", lw=1.2,
           label=f"Earth to orbit ≈ {DV_EARTH/1000:.1f} km/s  (required mass ratio = {R_earth:.1f})")
ax.set_yscale("log"); ax.set_xlim(0, 10); ax.set_ylim(0.4, 1e3)
ax.set_xlabel("Required ΔV [km/s]"); ax.set_ylabel("Liftoff mass [t] (log scale)")
ax.set_title("Liftoff mass to deliver a 500 kg capsule (Isp = 360 s)")
ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8.5, loc="lower right")
plt.tight_layout(); plt.show()

**(c) Report.** From the plot, report the required mass ratio for the Mars ascent and the maximum achievable mass ratio for $\varepsilon = 0.15$. Compare the two and state clearly whether the SSTO design is feasible for this mission, and why.

**Your answer:**

*Write your answer here.*

---
## 3.2 Two-Stage Mass Calculation [9 pts]

The cell below sizes the two-stage vehicle with your function, starting from Stage 2 (the kick stage, which carries the 500 kg capsule) and working backward to Stage 1 (the main ascent stage, which carries all of Stage 2 as its payload). It plots the mass breakdown of both architectures at the Mars $\Delta V$ as stacked bars. Each segment is labeled with its mass.

In [ ]:
# @title Provided code (run, do not edit): mass breakdown plot
_require_sec3()

def breakdown(m_pl, dv, eps):
    m0 = stage_initial_mass(m_pl, dv, ISP, eps)
    stage = m0 - m_pl                  # structure + propellant
    return m0, eps * stage, (1 - eps) * stage   # m0, m_s, m_p

S_m0, S_ms, S_mp = breakdown(M_CAPSULE, DV_MARS, EPS_SSTO)
B_m0, B_ms, B_mp = breakdown(M_CAPSULE, DV_MARS / 2, EPS_2STG)    # Stage 2 (kick)
A_m0, A_ms, A_mp = breakdown(B_m0, DV_MARS / 2, EPS_2STG)         # Stage 1 (ascent)

fig, ax = plt.subplots(figsize=(8.5, 6))
def seg(x, bottom, h, color, text):
    ax.bar(x, h, bottom=bottom, color=color, edgecolor="k", width=0.55)
    if h > 150:
        ax.text(x, bottom + h / 2, f"{text}\n{h:,.0f} kg", ha="center", va="center", fontsize=8.5)
    else:
        ax.text(x + 0.3, bottom + h / 2, f"{text}: {h:,.0f} kg", ha="left", va="center", fontsize=8.5)
    return bottom + h

b = seg(0, 0, M_CAPSULE, "0.85", "Capsule")
b = seg(0, b, S_ms, "C0", "Structure")
b = seg(0, b, S_mp, "lightsteelblue", "Propellant")
ax.text(0, b + 30, f"SSTO liftoff\n{S_m0:,.0f} kg", ha="center", va="bottom", fontweight="bold")

b = seg(1.2, 0, M_CAPSULE, "0.85", "Capsule")
b = seg(1.2, b, B_ms, "C1", "Stage 2 structure")
b = seg(1.2, b, B_mp, "navajowhite", "Stage 2 propellant")
ax.plot([0.75, 1.475], [B_m0, B_m0], "k:", lw=1)
ax.text(0.73, B_m0, f"Stage 2 initial mass\n{B_m0:,.0f} kg", ha="right", va="center", fontsize=8.5)
b = seg(1.2, b, A_ms, "C2", "Stage 1 structure")
b = seg(1.2, b, A_mp, "palegreen", "Stage 1 propellant")
ax.text(1.2, b + 30, f"Two-stage liftoff\n{A_m0:,.0f} kg", ha="center", va="bottom", fontweight="bold")

ax.set_xticks([0, 1.2]); ax.set_xticklabels(["SSTO (ε = 0.15)", "Two-stage (ε = 0.12 each)"])
ax.set_xlim(-0.4, 2.4); ax.set_ylim(0, max(S_m0, A_m0) * 1.15)
ax.set_ylabel("Mass [kg]"); ax.set_title("MAV mass breakdown at ΔV = 3,800 m/s")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

**Report.** From the plot, report the initial mass of Stage 2 and of Stage 1, and the total liftoff mass of the two-stage vehicle. Compare it with the SSTO liftoff mass: how many kilograms, and what percentage, does staging save?

**Your answer:**

*Write your answer here.*

---
## 3.3 Architecture Discussion [11 pts]

In **4–6 sentences**, discuss what these results mean for the MAV architecture trade. Your answer must address:
1. Why SSTO is feasible from Mars but would not be feasible from Earth with the same engine and structural fraction. Use the ceiling and the Earth line on the 3.1 plot, and your maximum-mass-ratio result.
2. The tradeoff between the mass saved by staging (from 3.2) and the added cost, complexity, and reliability risk of a staging event, and why this tradeoff is especially consequential for a Mars mission.

**Your answer:**

*Write your answer here.*